In [1]:
# Class 1: Sports & Athletics (Context: Winning/Medals)
doc1 = "The gold medal price is high effort"
doc2 = "Winning a gold medal needs a high jump"
doc3 = "Market for a gold medal is a trade of sweat"
doc4 = "The athlete will trade all for a gold medal"

# Class 2: Finance & Economy (Context: Market/Investment)
doc5 = "The gold bars price is high today"
doc6 = "Investing in gold bars needs a high rate"
doc7 = "Market for gold bars is a trade of money"
doc8 = "The bank will trade all for gold bars"

import numpy as np
from sklearn.cluster import KMeans


## Preprocessing Text + N-Gram Extraction + Vectorization

In [4]:
import contractions # type: ignore
from num2words import num2words # type: ignore
from sklearn.feature_extraction.text import CountVectorizer
import string
import spacy
nlp = spacy.load("en_core_web_sm")


def n2word(text : str):
  num = "".join(c for c in text if c.isdigit())
  if num: text = text.replace(num, num2words(int(num)))
  return text

def to_contraction(text : str):
  return contractions.fix(text.lower())

def drop_sw(tokens):
  return [t for t in tokens if not t.is_stop ]

def drop_p(text : str):
  translator = str.maketrans({key: " " for key in string.punctuation})
  return text.translate(translator)

def Stemming(tokens):
  return [t.lemma_ for t in tokens]



def preprocess_text(text):
  text = text.lower()
  text = to_contraction(text)
  text = n2word(text)
  return text

def vectorize(docs, n_gram_size=1):
  cleaned = []
  for doc in docs:
    text = preprocess_text(doc)
    text = drop_p(text)
    spacy_doc = nlp(text)
    tokens = drop_sw(spacy_doc)
    lemmas = Stemming(tokens)
    cleaned.append(' '.join(lemmas))

  cv = CountVectorizer(ngram_range=(1, n_gram_size))
  X  = cv.fit_transform(cleaned).toarray()
  return X, cv


## Training / Clustering

In [ ]:
all_docs = [doc1, doc2, doc3, doc4, doc5, doc6, doc7, doc8]

# 1-gram Experiment
X1, cv1 = vectorize(all_docs, n_gram_size=1)
km1 = KMeans(n_clusters=2, random_state=42).fit(X1)

# 2-gram Experiment
X2, cv2 = vectorize(all_docs, n_gram_size=2)
km2 = KMeans(n_clusters=2, random_state=42).fit(X2)

print(f"1-gram clusters: {km1.labels_}")
print(f"2-gram clusters: {km2.labels_}")



1-gram clusters: [0 1 0 0 1 1 0 0]
2-gram clusters: [0 0 0 0 1 1 1 1]


## Compare accuracy and precision using k-means

In [10]:
from sklearn.metrics import accuracy_score, precision_score

y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1])

def align_labels(pred: np.ndarray, true: np.ndarray) -> np.ndarray:
  """Flip cluster IDs if that produces a better accuracy match."""
  if accuracy_score(true, pred) >= accuracy_score(true, 1 - pred):
    return pred
  return 1 - pred

def evaluate(labels_raw: np.ndarray, y_true: np.ndarray, tag: str):
  labels = align_labels(labels_raw, y_true)
  acc  = accuracy_score(y_true, labels)
  prec = precision_score(y_true, labels, average='macro', zero_division=0)
  print(f"[{tag}]")
  print(f'Predicted: {labels.tolist()}')
  print(f'Ground-truth: {y_true.tolist()}')
  print(f'Accuracy: {acc:.2f}')
  print(f'Precision: {prec:.2f}')
  return acc, prec

acc1, prec1 = evaluate(km1.labels_, y_true, "1-gram BOW + KMeans")
print()
acc2, prec2 = evaluate(km2.labels_, y_true, "2-gram BOW + KMeans")



[1-gram BOW + KMeans]
Predicted: [0, 1, 0, 0, 1, 1, 0, 0]
Ground-truth: [0, 0, 0, 0, 1, 1, 1, 1]
Accuracy: 0.62
Precision: 0.63

[2-gram BOW + KMeans]
Predicted: [0, 0, 0, 0, 1, 1, 1, 1]
Ground-truth: [0, 0, 0, 0, 1, 1, 1, 1]
Accuracy: 1.00
Precision: 1.00


Accuracy in 2-gram BoW is much higher

## Task 2: Context Window

In [11]:
# Documents
D1 = "I love cats"
D2 = "Cats are chill"
D3 = "I am late"


def add_padding(tokens):
  # wrap tokens with start and end flags
  return ['<s>'] + tokens + ['</s>'] 


def extract_windows(tokens, window_size=1):  # slide window and collect all (2*window_size+1)-token windows
  wins = []
  pad_tokens = add_padding(tokens)
  window_length = 2 * window_size + 1
  
  for i in range(len(pad_tokens) - window_length + 1):
    window = pad_tokens[i : i + window_length]
    wins.append(" ".join(window))
  return wins


def build_vocab(all_windows):
  unique_wins = sorted(list(set(all_windows)))
  vocab = {window: idx for idx, window in enumerate(unique_wins)}
  return vocab

def vectorize_doc(doc_windows, vocab):
  vect = [0] * len(vocab)
  for win in doc_windows:
    if win in vocab: vect[vocab[win]] = 1
  return vect



docs = [D1, D2, D3]
all_extracted_windows = []
doc_windows_list = []

for doc in docs:
  tokens = doc.lower().split()
  windows = extract_windows(tokens, window_size=1)
  doc_windows_list.append(windows)
  all_extracted_windows.extend(windows)

vocab = build_vocab(all_extracted_windows)

vectors = []
for doc_windows in doc_windows_list:
  vectors.append(vectorize_doc(doc_windows, vocab))

print("Final Sorted Vocab Set:")
for window, idx in vocab.items():
  print(f"  {idx}: \"{window}\"")

print("\nDocument Vectors:")
for i, vec in enumerate(vectors):
  print(f"  D{i+1}: {vec}")



Final Sorted Vocab Set:
  0: "<s> cats are"
  1: "<s> i am"
  2: "<s> i love"
  3: "am late </s>"
  4: "are chill </s>"
  5: "cats are chill"
  6: "i am late"
  7: "i love cats"
  8: "love cats </s>"

Document Vectors:
  D1: [0, 0, 1, 0, 0, 0, 0, 1, 1]
  D2: [1, 0, 0, 0, 1, 1, 0, 0, 0]
  D3: [0, 1, 0, 1, 0, 0, 1, 0, 0]
